# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

STEMS = [
    "drums.wav",
    "vocals.wav",
    "bass.wav",
    "other.wav"
]
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0


In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    """
    Scans the genres_stems directory, filters out incomplete/corrupted songs,
    and returns train and val dictionaries.
    """
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    valid_songs = {g: [] for g in GENRES}

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue

        for song in sorted(os.listdir(genre_path)):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            stem_files = set(os.listdir(song_path))

            # COMPLETENESS: all expected stems must exist
            if not all(stem in stem_files for stem in STEMS):
                continue

            # CORRUPTION: any stem < 4KB => skip song
            corrupted = False
            for stem in STEMS:
                file_path = os.path.join(song_path, stem)
                try:
                    if os.path.getsize(file_path) < 4 * 1024:
                        corrupted = True
                        break
                except OSError:
                    corrupted = True
                    break

            if corrupted:
                continue

            valid_songs[genre].append(song_path)

    # Stratified shuffle split into train/val
    for genre in GENRES:
        songs = list(valid_songs.get(genre, []))
        rng.shuffle(songs)

        split_idx = int(len(songs) * (1 - val_split))
        train_songs = songs[:split_idx]
        val_songs = songs[split_idx:]

        # Helper to populate target dict
        def add_to_dict(target_dict, song_list):
            for song_path in song_list:
                for stem in STEMS:
                    stem_key = stem.replace('.wav', '')
                    target_dict[genre].setdefault(stem_key, [])
                    target_dict[genre][stem_key].append(os.path.join(song_path, stem))

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

In [5]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= threshold_sec
    """
    records = []
    total_files = 0

    for genre, stems in dataset_dict.items():
        for stem_name, files in stems.items():
            for file_path in files:

                total_files += 1

                # Load Audio
                y, _sr = librosa.load(file_path, sr=sr)
                total_duration = librosa.get_duration(y=y, sr=sr)

                # Find Non-Silent Intervals
                intervals = librosa.effects.split(y, top_db=top_db)

                max_silence = 0
                silence_type = []

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Full")

                else:
                    # CASE B: START silence
                    if intervals[0][0] > 0:
                        start_silence = intervals[0][0] / sr
                        max_silence = max(max_silence, start_silence)
                        silence_type.append("Start")

                    # CASE D: MIDDLE silence
                    for i in range(len(intervals) - 1):
                        middle_silence = (intervals[i+1][0] - intervals[i][1]) / sr
                        if middle_silence > 0:
                            max_silence = max(max_silence, middle_silence)
                            silence_type.append("Middle")

                    # CASE C: END silence
                    if intervals[-1][1] < len(y):
                        end_silence = (len(y) - intervals[-1][1]) / sr
                        max_silence = max(max_silence, end_silence)
                        silence_type.append("End")

                # Store result
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(sorted(set(silence_type))),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    print("Total files processed:", total_files)
    return df

# ----------------- execution order (must run) -----------------
tr, val = build_dataset(DATA_ROOT)

# compute silence dataframe
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

pivot_table = df_silence.pivot_table(
    index="Genre",
    columns="Stem",
    aggfunc="size",
    fill_value=0
)

print(pivot_table)

Total files processed: 3320
Stem       bass  drums  other  vocals
Genre                                
blues        17     22      5      43
classical    68     57      5      69
country      16     16      2      16
disco         8      2      3      18
hiphop       20      3     22       6
jazz         25     24      1      70
metal         6      2      1      40
pop          10      5      2       4
reggae        5      4      7      12
rock         10      7      1      26


In [6]:
stems_audio = []
try:
    desired_len = int(SR * DURATION)
    for key in STEM_KEYS:
        y, _ = librosa.load(
            tr[GENRE_TO_TEST][key][SONG_INDEX],
            sr=SR,
            duration=DURATION
        )
        # ensure equal length by padding/truncating
        y = librosa.util.fix_length(y, size=desired_len)

        stems_audio.append(y)

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")


Audio loaded successfully.


In [7]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

110250
0.1
0.59
1.0


In [8]:
def size_analysis(root_dir):
    corrupted_count = 0
    less_than_5_0491 = 0
    greater_than_5_0493 = 0

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            for stem in STEMS:
                file_path = os.path.join(song_path, stem)
                if not os.path.exists(file_path):
                    continue

                size_bytes = os.path.getsize(file_path)
                size_mb = size_bytes / (1024 * 1024)

                if size_bytes < 4 * 1024:
                    corrupted_count += 1

                if size_mb < 5.0491:
                    less_than_5_0491 += 1

                if size_mb > 5.0493:
                    greater_than_5_0493 += 1

    return corrupted_count, less_than_5_0491, greater_than_5_0493


corrupted, less_50491, greater_50493 = size_analysis(DATA_ROOT)

print("Q1 Answer:", corrupted + less_50491)
print("Q2 Answer:", abs(greater_50493 - less_50491))


Q1 Answer: 1256
Q2 Answer: 1072


In [11]:
train_reggae_drums = len(tr["reggae"]["drums"])
val_country_vocals = len(val["country"]["vocals"])

print("Q3 Answer:", abs(train_reggae_drums - val_country_vocals))

Q3 Answer: 66


In [12]:
# Q4
print("Q4 Answer:", len(df_silence))

# Q5
print("Q5 Answer:",
      df_silence[df_silence["Stem"]=="vocals"].shape[0])

# Q6
print("Q6 Answer:",
      round(df_silence[df_silence["Stem"]=="vocals"]["Max_Silence_Sec"].mean(), 2))

# Q7
print("Q7 Answer:",
      df_silence[
          (df_silence["Genre"]=="jazz") &
          (df_silence["Stem"]=="drums")
      ].shape[0])

# Q8
print("Q8 Answer:",
      df_silence[
          (df_silence["Genre"]=="jazz") &
          (df_silence["Stem"]=="drums") &
          (df_silence["Silence_Location"]=="Middle")
      ].shape[0])

# Q9
print("Q9 Answer:",
      df_silence[
          (df_silence["Genre"]=="jazz") &
          (df_silence["Stem"]=="drums") &
          (df_silence["Max_Silence_Sec"]>=10)
      ].shape[0])

print("Q10 Answer:", len(mix_raw))
print("Q11 Answer:", round(rms_val, 2))
print("Q12 Answer:", round(max_val, 2))


Q4 Answer: 680
Q5 Answer: 304
Q6 Answer: 12.59
Q7 Answer: 24
Q8 Answer: 0
Q9 Answer: 7
Q10 Answer: 110250
Q11 Answer: 0.1
Q12 Answer: 0.59
